In [1]:
from cmxflow import Block, SinkBlock, SourceBlock, Workflow, parameter

In [2]:
def reader(x):
    for i in range(10):
        yield i

def writer(x, y):
    for i in x:
        print(i)

class SimilarityBlock(Block):
    def __init__(self):
        super().__init__(input_files=["reference"])
        self.mutable(parameter.Categorical("fingerprint", "ecfp", ["ecfp", "rdk"]))

    def forward(self, x):
        return x ** 2

class SubstructureMatchBlock(Block):
    def __init__(self):
        super().__init__(input_text=["smarts"])
        self.mutable(parameter.Categorical("stereochemistry", True, [True, False]))

    def forward(self, x):
        return x ** 2

class MergeScoreBlock(Block):
    def forward(self, x):
        return x ** 2
    def check_input(self, x):
        return isinstance(x, int)

i = SourceBlock(reader)
b1 = SimilarityBlock()
b2 = SubstructureMatchBlock()
b3 = MergeScoreBlock()
o = SinkBlock(writer)
w = Workflow()
w.add(i, b1, b2, b3, o)
w

               ┌───────────────┐               
               │  SourceBlock  │               
               ├───────────────┤               
               │ input: [FILE] │               
               └───────────────┘               
                       ↓                       
 ┌───────────────────┐   ┌───────────────────┐ 
 │  SimilarityBlock  │   │   RequiredInput   │ 
 ├───────────────────┤ ← ├───────────────────┤ 
 │ fingerprint: ecfp │   │ reference: [FILE] │ 
 └───────────────────┘   └───────────────────┘ 
                       ↓                       
┌────────────────────────┐   ┌────────────────┐
│ SubstructureMatchBlock │   │ RequiredInput  │
├────────────────────────┤ ← ├────────────────┤
│ stereochemistry: True  │   │ smarts: [TEXT] │
└────────────────────────┘   └────────────────┘
                       ↓                       
              ┌─────────────────┐              
              │ MergeScoreBlock │              
              ├─────────────────┤       

In [3]:
w.get_required_input()

{'1.file@reference': str, '2.text@smarts': str}

In [4]:
required_input = {
    '1.file@reference': "test.csv",
    '2.text@smarts': "CCC",
}

w.set_required_input(required_input)
w

               ┌───────────────┐               
               │  SourceBlock  │               
               ├───────────────┤               
               │ input: [FILE] │               
               └───────────────┘               
                       ↓                       
┌───────────────────┐   ┌─────────────────────┐
│  SimilarityBlock  │   │    RequiredInput    │
├───────────────────┤ ← ├─────────────────────┤
│ fingerprint: ecfp │   │ reference: test.csv │
└───────────────────┘   └─────────────────────┘
                       ↓                       
┌────────────────────────┐   ┌───────────────┐ 
│ SubstructureMatchBlock │   │ RequiredInput │ 
├────────────────────────┤ ← ├───────────────┤ 
│ stereochemistry: True  │   │ smarts: CCC   │ 
└────────────────────────┘   └───────────────┘ 
                       ↓                       
              ┌─────────────────┐              
              │ MergeScoreBlock │              
              ├─────────────────┤       

In [5]:
w('a', 'b')

0
1
256
6561
65536
390625
1679616
5764801
16777216
43046721


In [5]:
from pathlib import Path

from cmxflow.sources import MoleculeSourceBlock

b = MoleculeSourceBlock()
b

┌────────────────┐
│ MoleculeSource │
├────────────────┤
│ input: [FILE]  │
└────────────────┘

In [6]:
list(b(Path("test.csv")))